# **Desafio Estatística com Python**
# Frequência e Medidas

### Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## **Configurações Iniciais**

Fonte dos dados: [Kaggle - Netflix Shows and Movies - Exploratory Analysis](https://www.kaggle.com/code/shivamb/netflix-shows-and-movies-exploratory-analysis)

In [ ]:
# Importa as bibliotecas
import pandas as pd
import numpy as np

# Carregamento da base de dados sobre catalogo da Netflix via Google Drive
# Armazena o id do arquivo csv e o insere na variavel da url
sheet_id = '1OJk-yF4ZgqyKP9Q6qgfWziiOQEOCV7mML0p0_AjdWGI'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'

# Le o csv e armazena os dados em um data frame
df = pd.read_csv(url)
df.head(0)

*Variáveis:*
* `show_id` - id único do filme/série.
* `title` - título do filme ou série
* `director` - diretor do filme ou série
* `cast` - elenco do filme ou série
* `country` - país do filme ou série
* `date_added` - data que foi adicionado no Netflix
* `release_year` - ano de lançamento original do filme
* `rating` - classificação da televisão
* `duration` - duração total do filme ou série.
* `listed_in` - categoria ou gênero do filme ou série.
* `description` - descrição do filme ou série.
* `type` - tipo de filme ou série

## **Exploração Inicial**

*   Quantas linhas e colunas tem o dataset?
*   Quais são os tipos das variáveis e se há valores ausentes?

## **Análises de Frequência**

* Qual a proporção de filmes vs. séries no catálogo?
* Qual o gênero mais frequente?

In [ ]:
# Analisa a proporcao de cada tipo de midia (filme ou serie)
total_tipo_midia = df['type'].value_counts() # contagem total de cada tipo de midia
total_tipo_midia_perc = df['type'].value_counts(normalize=True).round(2) * 100 # contagem percentual

# Junta o total e a proporcao de cada tipo de midia num unico df
df_tipo_midia= pd.concat([total_tipo_midia, total_tipo_midia_perc], axis=1)

# Renomeia as colunas
df_tipo_midia.index.name = 'Tipo de mídia'
df_tipo_midia.rename(columns={
    'count': 'Total',
    'proportion': 'Total em %'
  }, inplace=True)

# Imprime resposta
filmes_perc = df_tipo_midia['Total em %'].loc['Movie']
series_perc = df_tipo_midia['Total em %'].loc['TV Show']
print(f'O catálogo possui {filmes_perc}% de filmes e {series_perc}% de séries (razão: {filmes_perc / series_perc:.2f}/1).')

df_tipo_midia

In [ ]:
"""
Funcao que calcula a frequencia absoluta e percentual de uma Series e retorna um DataFrame.
"""
def calcular_frequencia(tabela, idx_txt):
  # Conta a quantidade absoluta de cada elemento
  df_resultado = tabela.value_counts().reset_index()

  # Renomeia as colunas para melhor legibilidade
  df_resultado.rename(columns={
      'index': idx_txt,
      'count': 'Total'
    }, inplace=True)
  # Redefine o indice para facilitar buscas futuras com .loc
  df_resultado.set_index(idx_txt, inplace=True)

  # Calcula as porcentagens, arrendondando para 2 casas
  df_resultado['Total em %'] = tabela.value_counts(normalize=True) * 100
  df_resultado['Total em %'] = df_resultado['Total em %'].round(2)

  return df_resultado

# Extrai cada genero listado em ´listed_in´: armazena em colunas
# separadas e depois empilha-as em uma unica coluna
generos_separados = df['listed_in'].str.split(', ', expand=True).stack()

# Gera o df com as contagens dos generos
idx_genero = 'Gênero'
df_generos = calcular_frequencia(generos_separados, idx_genero)

# Armazena o genero mais frequente
genero_moda = generos_separados.mode()[0]
# Imprime resposta
print(f'O gênero mais frequente é "{genero_moda}", presente em {df_generos["Total em %"].loc[genero_moda]}% do catálogo.')

df_generos.head()

In [ ]:
# Adicional - Limpeza dos dados: Remocao do tipo de midia
# O tipo de midia esta presente em alguns generos, o que pode distorcer a analise

# Atribui padrao aos elementos que tem apenas o tipo de midia
sem_genero = 'Sem Gênero'
# Dicionario mapeando os generos do catalogo para unificacao
mapeamento_genero = {
  'Movies': sem_genero,
  'TV Shows': sem_genero,
  'Docuseries': 'Documentaries',
  'Stand-Up Comedy & Talk Shows': 'Stand-Up Comedy,Talk Shows',
  'Classic & Cult TV': 'Classic,Cult',
}

# Substitui e separa os generos em novas linhas
df_generos_limpos = generos_separados.replace(mapeamento_genero).str.split(',').explode()

# Cria um dicionario para remover o tipo de midia dos generos
conteudo_limpeza = {}
# Adiciona os plurais dos tipos de midia
for tipo in df_tipo_midia.index:
  conteudo_limpeza[tipo + 's'] = ''

# Cria uma lista para remover termos redundantes e adiciona ao dicionario
termos_remover = ['TV', 'Series', 'Features']
for palavra in termos_remover:
  conteudo_limpeza[palavra] = ''

# Faz a limpeza dos dados, seguindo o dic e retirando espacos
df_generos_limpos = df_generos_limpos.str.replace(conteudo_limpeza,).str.strip()

# Armazena o genero mais frequente - precisa vir antes da funcao
moda_generos_limpos = df_generos_limpos.mode()[0]

# Gera o df com as contagens dos generos tratados
df_generos_limpos = calcular_frequencia(df_generos_limpos, idx_genero)

# Imprime resposta
perc_moda_generos_limpos = df_generos_limpos['Total em %'].loc[moda_generos_limpos]
print(f'Análise Tratada: Após a limpeza, o gênero mais frequente é "{moda_generos_limpos}", representando {perc_moda_generos_limpos}% do catálogo.')

df_generos_limpos.head()

## **Análises Estatísticas**

* Qual a média, mediana e moda do tempo de duração dos
filmes?
* Qual o filme mais curto e mais longo?

## **Visualização de Dados**

* Criar um gráfico de barras para mostrar a quantidade de títulos por gênero.
* Criar um histograma para analisar a distribuição da duração dos
filmes.

## **Atividade Extra**

* Quais são os 5 países que possuem mais produções no catálogo?


